## Insights 

#### Test hipothesis: By removing the "Deposit funds" banner we can keep users' attention focused on their "natural" behaviour: to find a stock to invest into and buy it, and therefore increase the D7 activation ratio.

- from statistical perspective there is no significant difference between both groups in making deposits, trading in 7 days after making a deposit and activating account in 7 days
- % of users in each group performing particular action is very similar (Metabase link)

## Recommendation 

##### Based on experiment performed, activation D7 and fund-to-trade D7 ratios did not improve when deposit banner was removed. Results for Control and Test groups are very similar and there is no statistical significance that one way is better then the other.

## Experiment

#### Overview:
- experiment details: JIRA TICKET
- experiment started on 31st May and lasted 14 days
- experiment variants: 
    - [1] Control (50% of all NEW users): these users continue seeing the “Deposit funds” banner
    - [2] Test (50% of all NEW users): these users will not see the “Deposit funds” banner
- users number in experiment: 918, users number in control group: 489, users number in test group: 429

#### Metrics:
- D7 Funded -> Traded, D7 Activation (Created -> Traded)

#### Significance:
No statistically significant difference observed for activation D7 and fund-to-trade D7 ratios between Control and Test groups

## Data Source

- User D7 fund-to-trade and activation data query - Metabase link

## 1. Preprocessing

### 1.1. Data and libraries import

In [ ]:
# import libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
import scipy.stats as stats
from itertools import product
%matplotlib inline

In [ ]:
# import data from csv file
df = pd.read_csv("ab_remove_deposit_banner.csv")
df['date_'] = pd.to_datetime(df['date_']).dt.date
df.head()

### 1.2 Data for number of users (daily and cumulated)

In [ ]:
# distinct users list
users_distinct = df[["user_id", "experiment_group", "date_"]].groupby(['user_id', 'experiment_group']).min().reset_index()

# check number of users in each experiment group
users_in_experiment = users_distinct.groupby("experiment_group").agg({"user_id":"count"}).reset_index()
users_in_experiment = users_in_experiment.rename(columns = {"user_id":"users_count"})

In [ ]:
# users assigned to experiment - trends
users_in_experiment_trends = users_distinct[["user_id", "experiment_group", "date_"]]\
                            .groupby(["experiment_group", "date_"]).agg({"user_id":"count"}).reset_index()
users_in_experiment_trends = users_in_experiment_trends.rename(columns = {"user_id" : "users_count"})

users_in_experiment_cumulated = users_in_experiment_trends.groupby(["experiment_group"]).agg({"users_count":"cumsum"}).reset_index()
users_in_experiment_cumulated = users_in_experiment_cumulated.rename(columns = {"users_count" : "users_cumulated"})

users_in_experiment_trends["users_cumulated"] = users_in_experiment_cumulated["users_cumulated"]

## 2. General insights

### 2.1. Experiment overview

In [ ]:
# display number of users in each experiment group
plt.figure(figsize=(8,4))
palette = "flare"
l1 = sns.barplot(data=users_in_experiment, x="experiment_group", y = "users_count")
plt.ylabel("Users count")
plt.xlabel("Experiment group")
plt.show()

In [ ]:
# plot users in experiment by day
plt.figure(figsize=(16,4))
palette = "husl"
l1 = sns.lineplot(data=users_in_experiment_trends, x="date_", y="users_count", hue="experiment_group")
plt.ylabel("Users count")
plt.xlabel("Day")
plt.title(f"Users assigned to experiment (daily)")
plt.show()

# plot users in experiment by day (cumulated)
plt.figure(figsize=(16,4))
palette = "husl"
l1 = sns.lineplot(data=users_in_experiment_trends, x="date_", y="users_cumulated", hue="experiment_group")
plt.ylabel("Users count")
plt.xlabel("Day")
plt.title(f"Users assigned to experiment (cumulated)")
plt.show()

### 2.2 Difference between test and control - trends

In [ ]:
# list of days to iterate
days_to_iterate = df["date_"].unique()

# metrics to be evaluated
metrics = ['fund_trade_d7', 'activated_d7']

In [ ]:
# run Mann-Whitney-U test: one sided for each day of expriment
dates_list = []
metrics_list = []
p_value_list = []
test_value = []
control_value = []
for i_date, j_metric in product(days_to_iterate, metrics):
    test_group = df[(df["experiment_group"]== "Test") & (df["date_"] == i_date)]
    control_group = df[(df["experiment_group"]== "Control") & (df["date_"] == i_date)]
    results = stats.mannwhitneyu(x=test_group[j_metric], y=control_group[j_metric], alternative = "greater", method = "auto", nan_policy = "omit").pvalue
    dates_list.append(i_date)
    metrics_list.append(j_metric)
    p_value_list.append(results)

In [ ]:
# create empty df
daily_p_value = pd.DataFrame(columns=["date", "metric", "p_value"])
# save results to df
daily_p_value = pd.DataFrame(data={"date":dates_list, "metric":metrics_list, "p_value":p_value_list})

In [ ]:
for i in metrics:
    plt.figure(figsize=(12,3))
    l1 = sns.lineplot(data=daily_p_value.loc[daily_p_value["metric"]== i], x="date", y='p_value', color = 'purple', label=i)
    l1.set_ylim(0,1.5)
    plt.xlabel(None)
    plt.xticks(size=6)
    plt.ylabel("p-value")
    #significance level
    plt.axhline(0.05)
    plt.show()